# Imports

In [1]:
from google.colab import files
import io
import os

import numpy as np
import pandas as pd

from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.callbacks import EarlyStopping

# Dataset

Load the pre-processed dataset.

In [2]:
# Remove file from colab environment if it already exists, to prevent multiple uploads when running code cell
!rm -rf HDB_Resale_Train.csv
!rm -rf HDB_Resale_Test.csv

# Create dictionary and store the data files
uploaded = files.upload()

# List all files and directories in the current working directory
print(os.listdir('.'))

# List files in uploaded
print(uploaded.keys())

# Train and test set
Train_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Train.csv']))
Test_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Test.csv']))

Saving HDB_Resale_Train.csv to HDB_Resale_Train.csv
Saving HDB_Resale_Test.csv to HDB_Resale_Test.csv
['.config', 'HDB_Resale_Train.csv', 'HDB_Resale_Test.csv', 'sample_data']
dict_keys(['HDB_Resale_Train.csv', 'HDB_Resale_Test.csv'])


### Encoding Categorical Features
For 'month_sold' feature:
- Perform cyclical encoding on the new column, which splits the column into two columns using sine and cosine:
    - sin(2π * month/12)
    - cos(2π * month/12)
- This models the feature in a cycle, meaning that 12 is as close to 1 as is to 11. This preserves the meaning of the month column.

For the rest of the categorical features:
- Perform one-hot-encoding

In [3]:
Train_df_encoded = Train_df.copy()

# Apply cyclical encoding to 'month_sold' feature
Train_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * Train_df['month_sold']/12)
Train_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * Train_df['month_sold']/12)

# Drop original month sold feature
Train_df_encoded = Train_df_encoded.drop(['month_sold'], axis=1)

categorical_cols = ['town', 'flat_type', 'storey_range', 'flat_model']

# Apply one-hot-encoding to the remaining catergorical features, with dtype=int for 0s and 1s
Train_df_encoded_final = pd.get_dummies(Train_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

display(Train_df_encoded_final.head())

,floor_area_sqm,resale_price,months_since_jan_2017,remaining_lease_years,month_sold_sin,month_sold_cos,town_ANG MO KIO,town_BEDOK,town_BISHAN,town_BUKIT BATOK,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,3.806662,12.354497,0,61.333333,0.5,0.866025,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,4.219508,12.429220,0,60.583333,0.5,0.866025,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,4.219508,12.476104,0,62.416667,0.5,0.866025,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,4.234107,12.487489,0,62.083333,0.5,0.866025,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,4.219508,12.487489,0,62.416667,0.5,0.866025,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [4]:
Test_df_encoded = Test_df.copy()

# Apply cyclical encoding to 'month_sold' feature
Test_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * Test_df['month_sold']/12)
Test_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * Test_df['month_sold']/12)

# Drop original month sold feature
Test_df_encoded = Test_df_encoded.drop(['month_sold'], axis=1)

# Apply one-hot-encoding to the remaining catergorical features, with dtype=int for 0s and 1s
Test_df_encoded_final = pd.get_dummies(Test_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

# Align columns for consistent feature sets between training and testing
# This ensures that if a category is present in training but not testing or vice versa,
# the dataFrames still have all columns, with missing ones filled with 0s.
aligned_columns = list(Train_df_encoded_final.columns)
Test_df_encoded_final = Test_df_encoded_final.reindex(columns=aligned_columns, fill_value=0)

display(Test_df_encoded_final.head())

,floor_area_sqm,resale_price,months_since_jan_2017,remaining_lease_years,month_sold_sin,month_sold_cos,town_ANG MO KIO,town_BEDOK,town_BISHAN,town_BUKIT BATOK,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,3.806662,12.691584,109,51.000000,0.866025,5.000000e-01,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3.806662,12.577640,111,50.833333,0.866025,-5.000000e-01,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3.806662,12.621491,111,50.750000,0.866025,-5.000000e-01,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3.828641,12.793862,111,58.833333,0.866025,-5.000000e-01,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,3.806662,12.706851,110,53.333333,1.000000,6.123234e-17,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Scaling Continuous Features

Scale the continuous features using StandardScaler(), ensuring the test set is transformed using the same scaler used for the training set.

In [5]:
# Define the continuous features to scale
continuous_cols = ['floor_area_sqm', 'resale_price', 'months_since_jan_2017', 'remaining_lease_years']

# Initialize the StandardScaler
scaler = StandardScaler()

Train_df_final = Train_df_encoded_final.copy()
Test_df_final = Test_df_encoded_final.copy()

# Fit the scaler on the training data's continuous features and transform them
Train_df_final[continuous_cols] = scaler.fit_transform(Train_df_final[continuous_cols])

# Transform the testing data's continuous features using the fitted scaler
Test_df_final[continuous_cols] = scaler.transform(Test_df_final[continuous_cols])

In [6]:
# Shuffle the training data to ensure the data is not in chronological order
Train_df_final = shuffle(Train_df_final, random_state=42)

# Reset index
Train_df_final = Train_df_final.reset_index(drop=True)

# Define features (X) and target (y)
X_train = Train_df_final.drop(columns=['resale_price'])
y_train = Train_df_final['resale_price']
X_test = Test_df_final.drop(columns=['resale_price'])
y_test = Test_df_final['resale_price']

# Global Single-Task Approach

This refers to the standard approach, a single ML model trained on the entire training data and tested on the entire testing data. Standard error metrics are used to evaluate performance. The model we will be using is a standard Multi-Layer Perceptron.

---

Neural networks are naturally stochastic, meaning certain operations are always random, such as:
- Model Weight Initialisation: Initial weights assigned to neurons are random.
- Dropout Layers: A set percentage but neurons chosen are random.
- Batches trained: The order is random.

Because of this, the model's results are different each time. By using a fixed random seed, the operations will be randomised in the same way each time. This will result in improved reproducibility (though results will still vary slightly).

---

The performance metrics used are:
1. Mean Squared Error (MSE) - Used for the loss function. MSE prioritises reducing the difference between predicted and actual values, so it this the best metric for house price prediction.
3. Root Mean Squared Error (RMSE) - Used as a performance metric to allow the results to be easily interpretable, it presents the loss in the same units as the target. However since the target variable was scaled, RMSE must be unscaled to be interpretable.
4. R-squared - Used as a performance metric to allow the results to be easily interpretable, it presents the percentage of variance explained by the model.

### Initial Model

Layers:
- Input layer
- 3 Dense layers
    - Neurons: 128 > 64 > 32 > 1
    - Activation: reLu
- 2 Dropout layers to prevent overfitting (Deactivates 20% of the neurons)
- Output layer
    - Neurons: 1
    - Activation: Linear (for regression)

Parameters:
- Learning rate: 0.001 (default for adam optimiser)
- Loss function: MSE
- Metric monitored: R-squared (can be understood immediately)
- Epochs: 30 (Large value to ensure the model converges)  
- Batch size: 32
- Validation set: 20% of training data
- Checkpoint: Instead of stopping training, the early stopping callback is used to save the best weights and restore them once training finishes

In [8]:
# VERY IMPORTANT: Use fixed seed for improved result reproducibility
tf.random.set_seed(42)

# Define the neural network model
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2), # Add dropout for regularisation
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2), # Add dropout for regularisation
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='linear') # Output layer for regression
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['r2_score'])

# Display model summary
model.summary()

# Create checkpoint callback by setting Earlystopping's patience value to more than the total epochs
Checkpoint = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[Checkpoint],
    verbose=1
)


# Evaluate the model on the test set
loss, r2 = model.evaluate(X_test, y_test, verbose=0)

# Make predictions on the test set
y_pred = model.predict(X_test).flatten()

# Calculate additional metrics
mse = loss
rmse = np.sqrt(mse)

print('Global Single-Task Approach Results:')
print('Mean Squared Error (MSE): ', mse)
print('Root Mean Squared Error (RMSE): ', rmse)
print('R-squared: ', round(r2 * 100, 2))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         9,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,225 (79.00 KB)

 Trainable params: 20,225 (79.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 45s 7ms/step - loss: 0.0887 - r2_score: 0.9112 - val_loss: 0.0672 - val_r2_score: 0.9331
Epoch 2/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 23s 4ms/step - loss: 0.0614 - r2_score: 0.9386 - val_loss: 0.0676 - val_r2_score: 0.9327
Epoch 3/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 0.0577 - r2_score: 0.9422 - val_loss: 0.0685 - val_r2_score: 0.9317
Epoch 4/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - loss: 0.0557 - r2_score: 0.9443 - val_loss: 0.0618 - val_r2_score: 0.9384
Epoch 5/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 0.0544 - r2_score: 0.9456 - val_loss: 0.0654 - val_r2_score: 0.9349
Epoch 6/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0534 - r2_score: 0.9466 - val_loss: 0.0633 - val_r2_score: 0.9369
Epoch 7/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - loss: 0.0526 - r2_score: 0.9474 - val_loss: 0.0650 - val_r2_score: 0.9352
Epoch 8/30
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.0520 - r2_score: 0

### Tuning

The initial model's performance is very good, with no signs of overfitting or underfitting based on the loss and val_loss. However, the val_loss seems to be fluctuating. In order to reduce that, we will try tuning the batch size.

In [9]:
# Initialise variables to store the results and predictions of the best model
best_batch_size = 32 # Batch size of the initial model
best_MSE = mse # MSE of the initial model
best_model_r2 = r2 # R-squared of the initial model
best_model_y_pred = y_pred # Predictions of the initial model

In [10]:
# Loop through the two learning rates
batch_size = [64, 128]

for batch in batch_size:
    print('Testing batch size of ', batch)

    # Define the neural network model
    model = keras.Sequential([
      keras.Input(shape=(X_train.shape[1],)),
      keras.layers.Dense(128, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(64, activation='relu'),
      keras.layers.Dropout(0.2), # Add dropout for regularisation
      keras.layers.Dense(32, activation='relu'),
      keras.layers.Dense(1, activation='linear') # Output layer for regression
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['r2_score'])

    # Create checkpoint callback by setting Earlystopping's patience value to more than the total epochs
    Checkpoint = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)

     # Train the model
    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=batch,
        validation_split=0.2,
        callbacks=[Checkpoint],
        verbose=0
    )

    # Evaluate the model on the test set
    loss, r2 = model.evaluate(X_test, y_test, verbose=0)

    # Make predictions on the test set
    y_pred = model.predict(X_test).flatten()

    mse = loss
    print('Mean Squared Error (MSE): ', mse)
    print()

    # If MSE is better, replace all results achieved by the best model
    if mse < best_MSE:
        best_batch_size = batch
        best_MSE = mse
        best_model_r2 = r2
        best_model_y_pred = y_pred
        print("New best model found with MSE: ", best_MSE)
        print()

Testing batch size of  64
177/177 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Mean Squared Error (MSE):  0.052462175488471985

New best model found with MSE:  0.052462175488471985

Testing batch size of  128
177/177 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Mean Squared Error (MSE):  0.06368964910507202



### Final Model

Display the results of the best model after tuning.

In [11]:
# Get the index of 'resale_price' in the continuous_cols list
resale_price_idx = continuous_cols.index('resale_price')

# Extract the mean and standard deviation for 'resale_price' from the fitted scaler
mean_resale_price = scaler.mean_[resale_price_idx]
std_resale_price = scaler.scale_[resale_price_idx]

# StandardScaler takes the values and subtracts the mean before dividing by the standard deviation
# To reverse scaling, multiply the scaled values by the standard deviation and then add the mean
y_pred_log = (best_model_y_pred * std_resale_price) + mean_resale_price
y_test_log = (y_test.values * std_resale_price) + mean_resale_price

# Reverse log transformation
y_pred_dollars = np.expm1(y_pred_log)
y_test_dollars = np.expm1(y_test_log)

# Calculate RMSE in dollars with the unscaled values
rmse = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

print('Global Single-Task Approach Final Results:')
print('Best batch size: ', best_batch_size)
print('Mean Squared Error (MSE): ', best_MSE)
print('Root Mean Squared Error (RMSE): ', np.sqrt(best_MSE))
print('RMSE in dollars: ', rmse)
print('R-squared: ', round(best_model_r2 * 100, 2))

Global Single-Task Approach Final Results:
Best batch size:  64
Mean Squared Error (MSE):  0.052462175488471985
Root Mean Squared Error (RMSE):  0.2290462300245782
RMSE in dollars:  60763.43350531256
R-squared:  93.64
